# 04 — LSTM Temporal Trend Prediction

Train LSTM on time-series glucose/acetone sequences to predict 24-hour metabolic trend.

**Inputs**: `data/kaggle/iot-sensor-diabetes/`, `data/kaggle/glucobench/`
**Output**: `apps/api/models/lstm_trend.keras`

**Metrics**: MAE, RMSE, trend accuracy (up/down/stable)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('../../..')
KAGGLE = ROOT / 'data' / 'kaggle'
PROCESSED = ROOT / 'data' / 'processed'
MODEL_DIR = ROOT / 'apps' / 'api' / 'models'

import warnings; warnings.filterwarnings('ignore')

## 1. Load IoT Sensor Diabetes (primary time-series)

In [ ]:
iot_files = list((KAGGLE / 'iot-sensor-diabetes').glob('*.csv'))
print('IoT files:', iot_files)
iot = pd.concat([pd.read_csv(f) for f in iot_files], ignore_index=True)
print('IoT shape:', iot.shape)
print(iot.columns.tolist())
iot.head()

In [ ]:
# Parse timestamp and sort
if 'Timestamp' in iot.columns:
    iot['ts'] = pd.to_datetime(iot['Timestamp'])
    iot = iot.sort_values(['Patient_ID', 'ts'])
    target_col = 'Glucose_Level'
elif 'timestamp' in iot.columns:
    iot['ts'] = pd.to_datetime(iot['timestamp'])
    iot = iot.sort_values(['Patient_ID', 'ts'])
    target_col = 'Glucose_Level'
else:
    target_col = iot.columns[2]  # fallback

print(f'Target column: {target_col}')
print(iot[target_col].describe())

## 2. Load GlucoBench (pre-training)

In [ ]:
gluco_files = list((KAGGLE / 'glucobench').glob('*.csv'))
print('GlucoBench files:', gluco_files)
if gluco_files:
    gluco = pd.concat([pd.read_csv(f) for f in gluco_files], ignore_index=True)
    print('GlucoBench shape:', gluco.shape)
    print(gluco.columns.tolist()[:10])

## 3. Build Sliding Window Sequences

In [ ]:
WINDOW = 24   # 24 timesteps input
HORIZON = 6   # predict 6 steps ahead

def build_sequences(df, patient_col, value_col, window=24, horizon=6):
    X_seqs, y_seqs = [], []
    for pid, group in df.groupby(patient_col):
        vals = group[value_col].fillna(method='ffill').values.astype(float)
        if len(vals) < window + horizon:
            continue
        for i in range(len(vals) - window - horizon + 1):
            X_seqs.append(vals[i:i+window])
            y_seqs.append(vals[i+window:i+window+horizon])
    return np.array(X_seqs), np.array(y_seqs)

X, y = build_sequences(iot, 'Patient_ID', target_col, WINDOW, HORIZON)
print(f'Sequences: X={X.shape}, y={y.shape}')

# Normalise
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_norm = scaler.fit_transform(X.reshape(-1, 1)).reshape(X.shape)
y_norm = scaler.transform(y.reshape(-1, 1)).reshape(y.shape)

# Reshape for LSTM: (samples, timesteps, features=1)
X_norm = X_norm[..., np.newaxis]

from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_norm, y_norm, test_size=0.15, random_state=42)
print(f'Train: {X_tr.shape} | Test: {X_te.shape}')

## 4. Build LSTM Model

In [ ]:
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)

model = keras.Sequential([
    keras.layers.Input(shape=(WINDOW, 1)),
    keras.layers.LSTM(64, return_sequences=True),
    keras.layers.Dropout(0.2),
    keras.layers.LSTM(32),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(HORIZON),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae'],
)
model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
]

history = model.fit(
    X_tr, y_tr,
    validation_split=0.15,
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    verbose=1,
)

## 5. Evaluation — MAE, RMSE, Trend Accuracy

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

y_pred_norm = model.predict(X_te)

# De-normalise
y_pred_actual = scaler.inverse_transform(y_pred_norm.reshape(-1, 1)).reshape(y_pred_norm.shape)
y_te_actual   = scaler.inverse_transform(y_te.reshape(-1, 1)).reshape(y_te.shape)

mae  = mean_absolute_error(y_te_actual.flatten(), y_pred_actual.flatten())
rmse = np.sqrt(mean_squared_error(y_te_actual.flatten(), y_pred_actual.flatten()))

# Trend accuracy: did we predict the correct direction?
true_trend  = np.sign(y_te_actual[:, -1]  - y_te_actual[:, 0])
pred_trend  = np.sign(y_pred_actual[:, -1] - y_pred_actual[:, 0])
trend_acc   = (true_trend == pred_trend).mean()

print(f'MAE:          {mae:.2f}')
print(f'RMSE:         {rmse:.2f}')
print(f'Trend Acc:    {trend_acc:.4f}')

# Plot training curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss (MSE)')
axes[0].legend()

# Sample prediction
idx = 0
axes[1].plot(range(WINDOW), scaler.inverse_transform(X_te[idx].reshape(-1,1)).flatten(), label='input')
axes[1].plot(range(WINDOW, WINDOW+HORIZON), y_te_actual[idx], label='true', linestyle='--')
axes[1].plot(range(WINDOW, WINDOW+HORIZON), y_pred_actual[idx], label='predicted')
axes[1].set_title(f'Sample Prediction (MAE={mae:.2f})')
axes[1].legend()

plt.tight_layout()
plt.savefig(MODEL_DIR / 'lstm_evaluation.png', dpi=150)
plt.show()

## 6. Save Model + Metrics

In [ ]:
import joblib, json

model.save(MODEL_DIR / 'lstm_trend.keras')
joblib.dump(scaler, MODEL_DIR / 'lstm_scaler.joblib')

metrics = {
    'mae': round(float(mae), 4),
    'rmse': round(float(rmse), 4),
    'trend_accuracy': round(float(trend_acc), 4),
    'window': WINDOW,
    'horizon': HORIZON,
    'n_train': len(X_tr),
    'n_test': len(X_te),
}
with open(MODEL_DIR / 'lstm_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved lstm_trend.keras + lstm_metrics.json')
print(metrics)